[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/34_speculative_decoding.ipynb)

# 🔴 Hard: Speculative Decoding

Implement the **acceptance/rejection step** of speculative decoding — a technique for accelerating LLM inference.

### Signature
```python
def speculative_decode(target_probs, draft_probs, draft_tokens) -> list[int]:
    # target_probs: (K, V) from target (large) model
    # draft_probs: (K, V) from draft (small) model
    # draft_tokens: (K,) tokens sampled by draft model
    # Returns: list of accepted tokens (1 to K)
```

### Algorithm
For each position i = 0, ..., K-1:
1. `ratio = target_probs[i, token_i] / draft_probs[i, token_i]`
2. Accept with probability `min(1, ratio)`
3. If rejected: sample from `normalize(max(0, target - draft))`, append, and stop

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch

/usr/local/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [27]:
# ✏️ YOUR IMPLEMENTATION HERE
def speculative_decode(target_probs, draft_probs, draft_tokens):
    n = len(draft_tokens)
    output = []
    for i in range(n):
        p = target_probs[i, draft_tokens[i]]
        q = draft_probs[i, draft_tokens[i]]
        sample_prob = min(1, p/q)
        if torch.rand(1) <= sample_prob:
            next_token = draft_tokens[i]
            output.append(next_token)
        else:
            diff = target_probs[i, :] - draft_probs[i, :]
            positive_diff = torch.clamp(diff, min=0.0)
            scaled_prob = positive_diff / torch.sum(positive_diff)
            next_token = torch.multinomial(scaled_prob,1)
            output.append(next_token)
            break
    return output

In [28]:
# 🧪 Debug
torch.manual_seed(0)
probs = torch.softmax(torch.randn(4, 10), dim=-1)
tokens = torch.tensor([2, 5, 1, 8])
print('Perfect draft:', speculative_decode(probs, probs, tokens))
target = torch.softmax(torch.randn(4, 10), dim=-1)
draft = torch.softmax(torch.randn(4, 10), dim=-1)
print('Random draft:', speculative_decode(target, draft, tokens))

Perfect draft: [tensor(2), tensor(5), tensor(1), tensor(8)]
Random draft: [tensor(2), tensor(5), tensor(1), tensor(8)]


In [29]:
# ✅ SUBMIT
from torch_judge import check
check('speculative_decoding')


🧪 Testing: Speculative Decoding (Hard)
──────────────────────────────────────────────────
  ✅ [1/3] Perfect draft: all accepted (2.4ms)
  ✅ [2/3] Output length bounded (1.4ms)
  ✅ [3/3] All tokens valid (26.4ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (30.2ms total)
  Progress saved. Run status() to see your dashboard.

